**1. Install packages**

In [ ]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/9

**2. Import libraries**

In [ ]:
import os

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

/tmp/ipykernel_7581/234906496.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


**3. Add your Groq API**

In [ ]:
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"

**4. Create a small knowledge base**

In [ ]:
documents = [
    Document(page_content="""
    Python is a popular programming language.
    It is easy to learn and widely used in AI, machine learning,
    web development, automation and data science.
    """),

    Document(page_content="""
    Machine Learning is a field of AI where computers learn patterns
    from data and use those patterns to make predictions or decisions.
    """),

    Document(page_content="""
    RAG stands for Retrieval-Augmented Generation.
    RAG allows an AI model to retrieve relevant information from
    external documents before generating an answer.
    """)
]

print("Documents created:", len(documents))

Documents created: 3


**5. Create embeddings**

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**6. Store documents in Chroma**

In [ ]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print("Vector database created!")

Vector database created!


**7. Create the retriever**

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

**8. Create the Groq model**

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

**9. Create the RAG chatbot**

In [ ]:
def rag_chat(question):

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved information
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not available in the context, say:
"I don't know based on the provided information."
"""

    response = llm.invoke(prompt)

    return response.content

**10. Test the chatbot**

In [ ]:
question = "What is RAG?"

answer = rag_chat(question)

print(answer)

RAG stands for Retrieval‑Augmented Generation. It allows an AI model to retrieve relevant information from external documents before generating an answer.


**11. Try another question**

In [ ]:
question = "What is Python used for?"

print(rag_chat(question))

Python is used for AI, machine learning, web development, automation, and data science.


**12. Simple chatbot loop**

In [ ]:
while True:

    question = input("\nYou: ")

    if question.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye!")
        break

    answer = rag_chat(question)

    print("Bot:", answer)


You: what is rag
Bot: RAG stands for Retrieval‑Augmented Generation. It is a technique that allows an AI model to retrieve relevant information from external documents before generating an answer.

You: how are you
Bot: I don't know based on the provided information.

You: what is my name
Bot: I don't know based on the provided information.

You: bye
Bot: Goodbye!
